# FNN Minimal Validation

This notebook validates the `cajal/fnn` code path in our project workspace without touching the multiscale coupling code.

The notebook supports two modes:
- preferred: load a pretrained MICrONS digital twin if its parameter directory already exists locally
- fallback: build the FNN architecture skeleton and randomize its parameters so we can still validate the tensor contract and runtime behavior

The fallback mode is only a technical validation. It is not biologically meaningful.


In [ ]:
from pathlib import Path
import sys


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src" / "fnn").exists() and (candidate / "notebook").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root.")


PROJECT_ROOT = find_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / "notebook"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from fnn_tvb_notebook_utils import ensure_src_on_sys_path, find_project_root as helper_find_project_root
ensure_src_on_sys_path(PROJECT_ROOT)
assert helper_find_project_root(PROJECT_ROOT) == PROJECT_ROOT

import numpy as np
import matplotlib.pyplot as plt


In [ ]:
from fnn_tvb_notebook_utils import (
    build_constant_covariates,
    build_random_fnn,
    load_pretrained_fnn,
    make_moving_bar_video,
    predict_responses,
    set_reproducible_seed,
)

set_reproducible_seed(7)

OUTPUT_DIR = PROJECT_ROOT / "notebook" / "output" / "fnn_validation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SESSION = 8
SCAN_IDX = 5
PRETRAINED_DIR = PROJECT_ROOT / "notebook" / "data" / "fnn" / "microns_digital_twin" / "params"
PRETRAINED_READY = (PRETRAINED_DIR / "params_core.pt").exists() and (PRETRAINED_DIR / f"params_{SESSION}_{SCAN_IDX}.pt").exists()
USE_PRETRAINED = PRETRAINED_READY

FRAME_COUNT = 90
N_UNITS = 64
FPS = 30.0

print("Project root:", PROJECT_ROOT)
print("Pretrained params dir exists:", PRETRAINED_DIR.exists())
print("Pretrained ready:", PRETRAINED_READY)


In [ ]:
frames = make_moving_bar_video(frame_count=FRAME_COUNT)
perspectives, modulations = build_constant_covariates(len(frames))

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
sample_indices = [0, len(frames) // 2, len(frames) - 1]
for ax, idx in zip(axes, sample_indices):
    ax.imshow(frames[idx], cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"Frame {idx}")
    ax.axis("off")
fig.suptitle("Synthetic moving-bar stimulus for FNN validation")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "fnn_validation_stimulus_frames.png", dpi=150, bbox_inches="tight")
plt.show()

print("Stimulus shape:", frames.shape, frames.dtype)


In [ ]:
if USE_PRETRAINED:
    if not PRETRAINED_DIR.exists():
        raise FileNotFoundError(
            f"Pretrained directory not found: {PRETRAINED_DIR}. "
            "Download it first or set USE_PRETRAINED = False."
        )
    model, unit_ids = load_pretrained_fnn(
        params_dir=PRETRAINED_DIR,
        session=SESSION,
        scan_idx=SCAN_IDX,
        cuda=False,
    )
    model_mode = f"pretrained MICrONS digital twin ({SESSION}-{SCAN_IDX})"
else:
    model = build_random_fnn(units=N_UNITS, randomize=True)
    unit_ids = None
    model_mode = f"randomized FNN skeleton with {N_UNITS} units"

responses = predict_responses(model, frames, perspectives, modulations)
total_params = sum(param.numel() for param in model.parameters())

print("Model mode:", model_mode)
print("Total parameters:", total_params)
print("Response shape:", responses.shape)
print("Response min/max:", float(responses.min()), float(responses.max()))
print("Response mean/std:", float(responses.mean()), float(responses.std()))


In [ ]:
mean_response = responses.mean(axis=1)
neuron_subset = slice(0, min(16, responses.shape[1]))

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(mean_response, color="black", lw=2)
axes[0].set_ylabel("Mean response")
axes[0].set_title("Population-mean FNN response")

im = axes[1].imshow(
    responses[:, neuron_subset].T,
    aspect="auto",
    origin="lower",
    cmap="viridis",
)
axes[1].set_xlabel("Frame")
axes[1].set_ylabel("Unit index")
axes[1].set_title("Example unit responses")
fig.colorbar(im, ax=axes[1], label="Predicted activity")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "fnn_validation_responses.png", dpi=150, bbox_inches="tight")
plt.show()


## Interpretation

If `USE_PRETRAINED = False`, this notebook validates that:
- the FNN architecture can be built inside the project environment
- the expected input shape is accepted: `frames x 144 x 256`, `uint8`
- recurrent forward prediction returns a `frames x units` response matrix

In fallback mode, the outputs are only useful as a technical smoke test.
For scientific use, switch to pretrained MICrONS parameters.
